In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("srcData1", "adlsrcset1")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")

In [0]:
container = dbutils.widgets.get("container")
srcData1 = dbutils.widgets.get("srcData1")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

ruta = f"abfss://{container}@{srcData1}.dfs.core.windows.net/clientes.csv"

In [0]:
df_circuits = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
schema_clientes = StructType(fields=[StructField("cliente_id", LongType(), False),
                                    StructField("edad", IntegerType(), True),
                                    StructField("ingreso_mensual", DoubleType(), True),
                                    StructField("antiguedad_bancaria", IntegerType(), True),
                                    StructField("departamento", StringType(), True),
                                    StructField("banco_principal", StringType(), True),
                                    StructField("app_preferida", StringType(), True),
                                    StructField("tipo_operacion_frecuente", StringType(), True),
                                    StructField("monto_promedio_transaccion", DoubleType(), True),
                                    StructField("tiene_token_fisico", IntegerType(), True),
                                    StructField("tiene_token_digital", IntegerType(), True),
                                    StructField("score_riesgo", DoubleType(), True),
                                    StructField("fecha_registro", StringType(), True)
])

In [0]:
df_clientes_final = spark.read\
.option('header', True)\
.schema(schema_clientes)\
.csv(ruta)

In [0]:
# DBTITLE 1, Selección de columnas de Clientes
clientes_selected_df = df_clientes_final.select(
    col("cliente_id"),
    col("edad"),
    col("ingreso_mensual"),
    col("antiguedad_bancaria"),
    col("departamento"),
    col("banco_principal"),
    col("app_preferida"),
    col("tipo_operacion_frecuente"),
    col("monto_promedio_transaccion"),
    col("tiene_token_fisico"),
    col("tiene_token_digital"),
    col("score_riesgo"),
    col("fecha_registro")
)

In [0]:
# DBTITLE 1, Agregar fecha de ingesta (Auditoría)
clientes_final_df = clientes_selected_df.withColumn("_fecha_ingesta", current_timestamp())

In [0]:
clientes_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.clientes")